# Mini-Projet : When Machine Learning Fails
## Analyse des limites : Masquage d'overfitting et défaut d'extrapolation sur le jeu de données Paddy

Ce notebook présente la démarche scientifique d'analyse des pannes de modèles de Machine Learning non-linéaires (ici la Forêt Aléatoire) sur des problématiques physiques.

### 1. Introduction et Chargement des Données
Nous commençons par charger le jeu de données `paddydataset.csv` et supprimons les doublons éventuels.

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import os

# Configuration visuelle des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [15, 5]

df = pd.read_csv('paddydataset.csv')
df = df.drop_duplicates()
df.columns = df.columns.str.strip()
print("Dimensions du dataset :", df.shape)

ModuleNotFoundError: No module named 'pandas'

### 2. Première Tentative : Provocation de l'Overfitting par Encodage Catégoriel
Nous essayons de forcer le sur-apprentissage en appliquant un One-Hot Encoding sur les variables catégorielles. 

In [ ]:
X = df.drop(columns=['Paddy yield(in Kg)'])
y = df['Paddy yield(in Kg)']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

# Pipeline d'expérience autonome
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features)
    ])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

pipeline.fit(X_train, y_train)
y_pred_train = pipeline.predict(X_train)
y_pred_test = pipeline.predict(X_test)

print(f"R2 Train : {r2_score(y_train, y_pred_train):.3f}")
print(f"R2 Test  : {r2_score(y_test, y_pred_test):.3f}")

# Génération du premier graphique
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred_test, alpha=0.6, color='#1f4e79', edgecolors='k')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_title("Tentative 1 : Valeurs Réelles vs Prédites\n(Split Aléatoire Standard)", fontsize=11, fontweight='bold')
ax.set_xlabel("Valeurs Réelles (Kg)", fontsize=10)
ax.set_ylabel("Valeurs Prédites (Kg)", fontsize=10)


### 3. Deuxième Tentative : Forçage de l'échec par Extrapolation (OOD Split)
Pour forcer l'échec de la forêt aléatoire, nous coupons les données selon la taille des parcelles : l'entraînement se fait sur de petites parcelles, le test sur de grandes parcelles. C'est un test Out-Of-Distribution (OOD).

**Symptôme :** Le modèle d'arbres est incapable d'extrapoler au-delà de la valeur cible maximale vue lors de l'entraînement.

In [ ]:
median_hectares = df['Hectares'].median()
df_train = df[df['Hectares'] <= median_hectares].copy()
df_test = df[df['Hectares'] > median_hectares].copy()

X_train_ood = df_train.drop(columns=['Paddy yield(in Kg)'])
y_train_ood = df_train['Paddy yield(in Kg)']
X_test_ood = df_test.drop(columns=['Paddy yield(in Kg)'])
y_test_ood = df_test['Paddy yield(in Kg)']

numeric_features_ood = X_train_ood.select_dtypes(include=['int64', 'float64']).columns
categorical_features_ood = X_train_ood.select_dtypes(include=['object', 'category']).columns

preprocessor_ood = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features_ood),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features_ood)
    ])

pipeline_ood = Pipeline(steps=[
    ('preprocessor', preprocessor_ood),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

pipeline_ood.fit(X_train_ood, y_train_ood)
y_pred_train_ood = pipeline_ood.predict(X_train_ood)
y_pred_test_ood = pipeline_ood.predict(X_test_ood)

print(f"R2 Train OOD : {r2_score(y_train_ood, y_pred_train_ood):.3f}")
print(f"R2 Test OOD  : {r2_score(y_test_ood, y_pred_test_ood):.3f}")

# Génération du deuxième graphique
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test_ood, y_pred_test_ood, alpha=0.6, color='#c00000', edgecolors='k')
ax.plot([y_test_ood.min(), y_test_ood.max()], [y_test_ood.min(), y_test_ood.max()], 'r--', lw=2)
ax.set_title("Tentative 2 : Valeurs Réelles vs Prédites\n(Split OOD - Échec d'Extrapolation)", fontsize=11, fontweight='bold')
ax.set_xlabel("Valeurs Réelles (Kg)", fontsize=10)
ax.set_ylabel("Valeurs Prédites (Kg)", fontsize=10)
plt.tight_layout()

### 4. Correction de la Défaillance : Approches Naïve et Robuste

Nous étudions deux stratégies de correction :
1. **Correction Naïve (Target Scaling Seul) :** On entraîne le modèle à prédire le taux spécifique $Yield/Hectares$, puis on multiplie la prédiction par $Hectares$. Le $R^2$ de test s'élève à $0.706$. 
   * *La défaillance cachée :* Les descripteurs d'intrants physiques de taille (comme `Seedrate`) n'étant pas mis à l'échelle, ils sont hors-distribution sur le jeu de test. La forêt clippe ces valeurs, prédisant un taux presque constant (corrélation nulle/négative de **-0.11** avec le taux de rendement réel). Le score de 0.706 est un artefact mathématique porté par la multiplication par la surface.
2. **Correction Robuste (Pipeline Invariant d'Échelle) :** Pour corriger proprement la défaillance d'extrapolation sur les intrants, on normalise tous les intrants par rapport aux Hectares, et on supprime la variable `Hectares` des prédicteurs. Le modèle apprend la pure relation de productivité agronomique ($R^2$ de test sur le taux de $0.380$ sans illusion d'échelle).

In [ ]:
from data_loader import get_scale_dependent_features, scale_features_by_hectares

# ----------------------------------------------------
# 1. Correction Naïve (Target Scaling Seul)
# ----------------------------------------------------
y_train_rate = y_train_ood / X_train_ood['Hectares']

pipeline_corr_naive = Pipeline(steps=[
    ('preprocessor', preprocessor_ood),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

pipeline_corr_naive.fit(X_train_ood, y_train_rate)
y_pred_rate_naive = pipeline_corr_naive.predict(X_test_ood)
y_pred_test_naive = y_pred_rate_naive * X_test_ood['Hectares']

r2_test_naive = r2_score(y_test_ood, y_pred_test_naive)
y_test_rate = y_test_ood / X_test_ood['Hectares']
corr_rate_naive = np.corrcoef(y_pred_rate_naive, y_test_rate)[0, 1]

print(f"[A] Correction Naïve (Target Scaling seul) :")
print(f"  R2 Test Naïf : {r2_test_naive:.3f}")
print(f"  Corrélation Taux Prédit vs Réel : {corr_rate_naive:.3f}")

# ----------------------------------------------------
# 2. Correction Robuste (Scale-Invariant Pipeline)
# ----------------------------------------------------
scale_cols = get_scale_dependent_features(df)
X_train_scaled = scale_features_by_hectares(X_train_ood, scale_cols).drop(columns=['Hectares'])
X_test_scaled = scale_features_by_hectares(X_test_ood, scale_cols).drop(columns=['Hectares'])

numeric_features_scaled = X_train_scaled.select_dtypes(include=['int64', 'float64']).columns
categorical_features_scaled = X_train_scaled.select_dtypes(include=['object', 'category']).columns

preprocessor_robust = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features_scaled),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features_scaled)
    ])

pipeline_robust = Pipeline(steps=[
    ('preprocessor', preprocessor_robust),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

pipeline_robust.fit(X_train_scaled, y_train_rate)
y_pred_rate_robust = pipeline_robust.predict(X_test_scaled)
y_pred_test_robust = y_pred_rate_robust * X_test_ood['Hectares']

r2_test_robust = r2_score(y_test_ood, y_pred_test_robust)
corr_rate_robust = np.corrcoef(y_pred_rate_robust, y_test_rate)[0, 1]

print(f"\n[B] Correction Robuste (Scale-Invariant) :")
print(f"  R2 Test Robuste : {r2_test_robust:.3f}")
print(f"  Corrélation Taux Prédit vs Réel : {corr_rate_robust:.3f}")

# Génération des graphiques
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot A : Correction Naïve (valeurs brutes)
axes[0].scatter(y_test_ood, y_pred_test_naive, alpha=0.6, color='#ff9800', edgecolors='k')
axes[0].plot([y_test_ood.min(), y_test_ood.max()], [y_test_ood.min(), y_test_ood.max()], 'r--', lw=2)
axes[0].set_title(f"A: Correction Naïve (Target Scaling seul)\nR2 = {r2_test_naive:.3f} | Corr Taux = {corr_rate_naive:.3f}", fontsize=10, fontweight='bold')
axes[0].set_xlabel("Valeurs Réelles (Kg)")
axes[0].set_ylabel("Valeurs Prédites (Kg)")

# Plot B : Correction Robuste (Scale-Invariant)
axes[1].scatter(y_test_ood, y_pred_test_robust, alpha=0.6, color='#2e7d32', edgecolors='k')
axes[1].plot([y_test_ood.min(), y_test_ood.max()], [y_test_ood.min(), y_test_ood.max()], 'r--', lw=2)
axes[1].set_title(f"B: Correction Robuste (Scale-Invariant)\nR2 = {r2_test_robust:.3f} | Corr Taux = {corr_rate_robust:.3f}", fontsize=10, fontweight='bold')
axes[1].set_xlabel("Valeurs Réelles (Kg)")
axes[1].set_ylabel("Valeurs Prédites (Kg)")

plt.tight_layout()

### 5. Clé de lecture des différents graphiques du notebook (Légende explicative)

1. **Graphique de la Tentative 1 (Split Aléatoire standard) :** Montre un alignement presque parfait autour de la bissectrice (modèle à forte performance $R^2 \approx 0.989$). Le modèle n'a pas sur-appris sur les variables catégorielles grâce au raccourci d'apprentissage sur la surface.
2. **Graphique de la Tentative 2 (Split OOD) :** Les prédictions plafonnent sur une ligne horizontale stricte correspondant au rendement maximum vu en entraînement. C'est le symptôme typique du manque d'extrapolation des arbres.
3. **Graphique de la Correction A (Naïve) :** Les points s'étalent le long de la diagonale, mais l'analyse de corrélation révèle que le modèle prédit un taux fixe en raison du clipping des intrants OOD. C'est une illusion mathématique portée par `Hectares`.
4. **Graphique de la Correction B (Robuste) :** Les prédictions sont exemptes de fuite ou de clipping d'échelle. Le modèle capture sainement l'influence agronomique.